# 02 — Baselines and the paper benchmark (E01–E03)

This notebook **reads** results; it trains nothing. Every run was made by
`scripts/run_experiment.py` from a committed config (`configs/experiments/E0[1-3]_*.yaml`,
written by `scripts/make_configs.py`) and logged to MLflow; `scripts/export_history.py` exported
the runs to `reports/history.csv`, which is what this notebook loads. The same tables, in text,
are in `reports/baseline_tables.md` (`scripts/results_tables.py`).

Two protocols, never mixed:

- **Holdout (E01–E03, 80 runs):** data v1, stratified 70/15/15 split with seed 42, 5-fold CV on
  the training set, fit on train, score on validation. The test set is not touched.
- **Paper benchmark (64 runs, inside E02 and E03, tag `protocol=paper_10fold`):** the paper's own
  protocol, 10-fold stratified CV on the whole file with predictions pooled over the folds (as Weka
  reports them). Run on two row sources: `published` = every row of the ARFF file as published
  (duplicates kept, values untouched; column names from COLUMN_MAP, which doesn't change what a
  model sees) and `v1` = all rows of data v1 (deduplicated). These scores are never used to choose
  a model.

**Weka defaults used by the paper, and our stand-ins** (checked 2026-10-28 in the Weka Javadoc):

| Weka | Default | Source | Our scikit-learn stand-in |
|---|---|---|---|
| J48 (C4.5) | `-C 0.25` (pruning confidence), `-M 2` (min. instances per leaf), pruning and subtree raising on | https://weka.sourceforge.io/doc.stable/weka/classifiers/trees/J48.html | `DecisionTreeClassifier(criterion="entropy", min_samples_leaf=2, random_state=42)`; no equivalent of C4.5's pessimistic pruning, so the tree is unpruned |
| IBk (KNN) | `-K 1`, no distance weighting, `LinearNNSearch` with `EuclideanDistance` | https://weka.sourceforge.io/doc.stable/weka/classifiers/lazy/IBk.html | `KNeighborsClassifier(n_neighbors=1)` |
| EuclideanDistance | attributes normalised to their range unless `-D` is set ("default false i.e. attribute values are normalized") | https://weka.sourceforge.io/doc.stable/weka/core/EuclideanDistance.html, https://weka.sourceforge.io/doc.stable/weka/core/NormalizableDistance.html | `MinMaxScaler` fitted on the training folds, inside the pipeline |

The default distance of `LinearNNSearch` (`-A`, default `weka.core.EuclideanDistance`) is listed in
the Weka Javadoc of `weka.core.neighboursearch` (e.g. FilteredNeighbourSearch's option list).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from blindspy import results as R
from blindspy.data import CORE_TASKS, ROOT, TIMEOUTS

pd.set_option("display.width", 200, "display.max_columns", 30)
FIG = ROOT / "reports" / "figures"
h = R.load_history()
b = R.baselines(h)
bench = R.benchmark(h)
paper = R.load_paper()
commits = sorted(set(b["tags.git_commit"].str[:7]))
print(
    len(b),
    "baseline runs;",
    len(bench),
    "benchmark runs; data versions:",
    sorted(set(b["tags.data_version"])),
    "; commits:",
    commits,
)

## 1. Holdout baselines (E01–E03): validation macro F1

In [ ]:
R.score_grid(b, "metrics.val_f1_macro")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
colors = {
    "majority": "#9e9e9e",
    "tree": "#1f77b4",
    "knn": "#ff7f0e",
    "logreg": "#2ca02c",
    "svm_rbf": "#d62728",
}
w = 0.16
for ax, task in zip(axes, CORE_TASKS, strict=True):
    for i, m in enumerate(R.BASELINE_MODELS):
        s = b[(b["params.task"] == task) & (b["params.model"] == m)].sort_values("params.timeout")
        x = [TIMEOUTS.index(t) + (i - 2) * w for t in s["params.timeout"]]
        ax.bar(x, s["metrics.val_f1_macro"], w, color=colors[m], label=m)
        ax.errorbar(
            x,
            s["metrics.cv_f1_macro_mean"],
            yerr=s["metrics.cv_f1_macro_std"],
            fmt="none",
            ecolor="black",
            elinewidth=0.8,
            capsize=2,
        )
    ax.set_xticks(range(4), [f"{t} s" for t in TIMEOUTS])
    ax.set_title(task)
    ax.set_ylim(0, 1)
axes[0].set_ylabel("macro F1")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, fontsize=8)
fig.suptitle(
    "Baselines E01-E03, data v1: bars = validation macro F1, "
    "whiskers = 5-fold CV mean +- std (test untouched)"
)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(FIG / "baseline_model_comparison.png", dpi=120)

**Findings (E01–E03, validation set, data v1).**

- The entropy decision tree is the best baseline in all 16 configs: validation macro F1 0.706–0.890
  (A1 0.857–0.890; A2-VPN 0.775–0.828; A2-NonVPN 0.825–0.839; B 0.706–0.759).
- KNN (k = 5, standard scaling) is second everywhere (0.617–0.825) and within 0.10 of the tree in
  15 of 16 configs; the exception is B 30 s (0.647 vs 0.748).
- Logistic Regression (0.270–0.598) and SVM-RBF (0.274–0.621) are far behind the tree in every
  config, but well above the majority floor (gain at least 0.18).
- Task B (14 classes) is the hardest and A1 the easiest; within a task, the 60 s and 120 s files
  score lower for B and A2-VPN. Class mix differs between timeouts (E00), and the 60 s files hold
  differently shaped VOIP and FT flows (E00 addendum), so this is **not** read as a timeout effect.
- CV on the training set predicts validation well: |CV mean − validation| ≤ 0.054 for every run,
  and CV standard deviations are at most 0.025, so differences of a few points between models are
  larger than the fold-to-fold noise.

## 2. E01 check: the floor is exactly what arithmetic predicts

In [ ]:
R.majority_check(b)

**Finding.** In all 16 configs the majority model's validation accuracy equals the largest class's
share p and its macro F1 equals 2p / ((1 + p) k) to 4 decimals, as expected. This confirms the
metrics code and the stratified split, and fixes the floor: macro F1 0.34–0.36 for A1, 0.06–0.10
for A2 and 0.02–0.03 for B.

## 3. Per-class view, confusion pairs, complexity and cost

In [ ]:
for m in ("tree", "knn"):
    print(m)
    display(R.class_extremes(b, m))

In [ ]:
for m in ("tree", "knn"):
    print(m)
    display(R.confusion_pairs(b, m))

In [ ]:
display(R.complexity(b))
display(R.cost_table(b))

**Findings.**

- VoIP is the easiest traffic type (tree F1 0.92–0.99 for VOIP and VPN-VOIP); the weakest classes
  are CHAT, VPN-CHAT, STREAMING, MAIL and VPN-MAIL (tree F1 0.42–0.71).
- In B, the largest confusion for the tree and KNN in every timeout is BROWSING vs VPN-BROWSING
  (in both directions), i.e. the model knows it is browsing but not whether it went through the VPN.
  In A2 the main confusions are CHAT, FT and BROWSING with each other.
- Logistic Regression and SVM-RBF never predict MAIL / VPN-MAIL correctly in most A2 and B configs
  (F1 0.00), which is why their macro F1 is so much lower than their weighted F1 (gap up to 0.225).
  The tree's gap is 0.000–0.069.
- The unpruned tree grows to depth 16–35 with 591–3,437 nodes; the SVM keeps 55–88% of the
  training rows as support vectors, a sign that the RBF kernel on standard-scaled heavy-tailed
  features separates the classes poorly.
- Cost: the tree predicts 1,000 flows in 0.3 ms (median), KNN in 7.9 ms and the SVM in 211 ms;
  model files range from 1.4 KB (majority) and 4–5 KB (Logistic Regression) to 4.2 MB (SVM).
  Training times were measured with tracemalloc on, which slows fitting (x1.05 tree, x1.00 SVM,
  x1.19 Logistic Regression, x2.3 KNN), so they are slight overestimates. All of these
  train in under 3 s.

## 4. Paper benchmark, A1: precision and recall per class next to the paper

Paper values are read from Figure 2 to +-0.01 (`reports/paper_baseline.csv`, `source =
read_from_chart`); the values the text also states exactly (15 s and 120 s precision, marked
"chart +-0.01 and text") agree with them.
`diff` = ours - paper.

In [ ]:
a1 = R.paper_a1(bench, paper)
a1

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for (r, cls), (c, metric) in [
    ((0, "VPN"), (0, "precision")),
    ((0, "VPN"), (1, "recall")),
    ((1, "Non-VPN"), (0, "precision")),
    ((1, "Non-VPN"), (1, "recall")),
]:
    ax = axes[r][c]
    for algo, col in (("C4.5", "#1f77b4"), ("KNN", "#ff7f0e")):
        sel = (a1["class"] == cls) & (a1["metric"] == metric) & (a1["algorithm"] == algo)
        s = a1[sel].sort_values("timeout")
        x = [TIMEOUTS.index(t) for t in s["timeout"]]
        ax.errorbar(
            x, s["paper"], yerr=0.01, color=col, marker="o", lw=2, capsize=3, label=f"paper {algo}"
        )
        ax.plot(
            x,
            s["ours_published"],
            color=col,
            marker="s",
            ls="--",
            label=f"ours {algo}, published rows",
        )
        ax.plot(x, s["ours_v1"], color=col, marker="^", ls=":", label=f"ours {algo}, data v1")
    ax.set_title(f"{cls} {metric}")
    ax.set_xticks(range(4), [f"{t} s" for t in TIMEOUTS])
    ax.set_ylim(0.6, 1.0)
axes[0][0].legend(fontsize=7)
fig.suptitle("A1, paper protocol (10-fold CV): paper (Fig. 2) vs our C4.5 approximation and 1-NN")
fig.tight_layout()
fig.savefig(FIG / "paper_vs_ours_A1.png", dpi=120)

**Findings (A1, paper protocol).**

- On the rows as published, our C4.5 approximation matches all 16 of the paper's A1 chart values
  within ±0.05 (largest difference 0.021, close to the ±0.01 reading error of the chart).
- 1-NN with Weka-style min-max scaling matches 15 of 16 within ±0.05; the miss is VPN recall at
  120 s (0.803 vs 0.860).
- So the paper's A1 results are reproduced with our corrected column names and our own split of the
  same files. On data v1 (duplicates removed) the values move by at most 0.053.

## 5. Paper benchmark, A2 and B: the values the paper states in its text

Figure 3 draws two unlabelled lines per timeout, so per-class A2/B values can't be assigned to
C4.5 or KNN; only the numbers written in the text are compared. For averages the paper doesn't say
whether it is a plain or class-weighted mean (Weka prints the weighted one), so both are shown.

In [ ]:
R.paper_text_values(bench, paper)

**Findings (A2 and B, values stated in the paper's text).**

- A2, C4.5 at 15 s: the paper's average precision (0.84 VPN, 0.89 non-VPN) is matched by our
  macro-averaged precision within 0.008 on the published rows.
- B: the best-timeout average precision is matched within 0.013 (C4.5) and 0.001 (KNN) by the
  macro average, and less well by the weighted average (+0.025, +0.034). This suggests the paper's
  "average" is a plain (macro) mean, but the paper doesn't say so.
- Per-class B values at 15 s are matched within 0.04 on published rows, except where the paper's
  number is far from ours at 120 s for the mail classes (C4.5 MAIL 0.68 vs 0.79; KNN VPN-MAIL
  0.82 vs 0.69). Mail classes are small, so one fold assignment moves them a lot.

## 5b. Paper benchmark: one table per task

Our 10-fold precision and recall for every class, both algorithms and all timeouts, with the paper's
value in brackets wherever it gives one (C = read from its chart, +-0.01; T = stated in its text).
First on the rows as published, then on data v1.

In [ ]:
for rows in ("published", "v1"):
    for task in CORE_TASKS:
        print(task, "rows:", rows)
        display(R.paper_task_table(bench, paper, task, rows))

## 6. Paper benchmark: what the duplicates do to 10-fold scores

In [ ]:
d = R.dedup_effect(bench)
display(d)
fig, ax = plt.subplots(figsize=(14, 3.8))
labels = [f"{t}\n{s}s" for t in CORE_TASKS for s in TIMEOUTS]
for i, (algo, col) in enumerate((("j48", "#1f77b4"), ("ibk", "#ff7f0e"))):
    s = d[d["algo"] == algo]
    ax.bar(
        [k + (i - 0.5) * 0.4 for k in range(16)],
        s["v1_minus_published"],
        0.4,
        color=col,
        label=f"{R.BENCH_ALGO[algo]} ({algo})",
    )
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(16), labels, fontsize=7)
ax.set_ylabel("macro F1: v1 - published")
ax.set_title(
    "10-fold macro F1 change when exact duplicates are removed (data v1 vs rows as published)"
)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "benchmark_dedup_effect.png", dpi=120)

In [ ]:
R.duplicate_profile()

**Finding: the pre-stated hypothesis (E02 d, E03 e) is not supported.** Removing the exact
duplicates did not lower the 10-fold scores: macro F1 rose in 10 of 16 files for the tree and 14 of
16 for 1-NN (mean change +0.001 and +0.012).

The table above shows why. The duplicates are not copies of ordinary flows. 39–92% of the removed
rows have an identical twin with a **different** label, against 2–10% of all rows, and their median
flow has 0–2 packets (all flows: median 8–43; A1 120 s left out, its packet column is the corrupt published one). They are degenerate, near-empty flows that timing
features can't tell apart. Keeping every copy adds more of these unavoidable errors, so removing
them helps slightly. The leakage risk that motivated the duplicate rule (a copy of a test flow in the
training data) is real, but here it is outweighed by this effect. The rule stays (it is right for an
honest split), and this result goes in the report.

## 7. Threshold checks (pre-stated in E01–E03)

In [ ]:
v = R.verdicts(b, bench, paper)
pd.DataFrame([{"check": k, **x} for k, x in v.items()])

**Summary of the pre-stated thresholds:** E01 supported (16/16). E02 (a) and (b) supported, (c)
supported (16/16 A1 values within ±0.05), (d) not supported (6/16 files lower on v1). E03 (a) supported
for all three models, (b) supported (15/16), (c) supported for both (16/16), (d) supported (15/16),
(e) not supported (2/16).